In [2]:
# ==============================
# 1. КОНТРАСТНАЯ ДИСТИЛЛЯЦИЯ - IMPORTS
# ==============================
import os, sys, math, random, json, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from glob import glob
from PIL import Image
from collections import defaultdict
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ==============================
# 2. CONTRASTIVE LOSS (InfoNCE)
# ==============================
class ContrastiveLoss(nn.Module):
    """
    Контрастный loss для дистилляции признаков.
    Использует InfoNCE (NT-Xent) для приближения embedding'ов студента к учителю.
    
    Args:
        temperature: температура для softmax
        normalize: нормализовать ли векторы перед вычислением similarity
    """
    def __init__(self, temperature=0.07, normalize=True):
        super().__init__()
        self.temperature = temperature
        self.normalize = normalize
        
    def forward(self, student_features, teacher_features):
        """
        student_features: [batch_size, dim_student]
        teacher_features: [batch_size, dim_teacher]
        """
        batch_size = student_features.shape[0]
        
        # Нормализация
        if self.normalize:
            student_features = F.normalize(student_features, dim=1)
            teacher_features = F.normalize(teacher_features, dim=1)
        
        # Вычисляем similarity matrix: [batch_size, batch_size]
        # Каждый студент должен быть близок к соответствующему учителю (positive pair)
        # и далек от других (negative pairs)
        similarity_matrix = torch.matmul(student_features, teacher_features.T) / self.temperature
        
        # Labels: диагональные элементы - positive pairs
        labels = torch.arange(batch_size).to(student_features.device)
        
        # InfoNCE loss (cross-entropy)
        loss = F.cross_entropy(similarity_matrix, labels)
        
        return loss

# ==============================
# 3. COMBINED CONTRASTIVE + CLASSIFICATION LOSS
# ==============================
class ContrastiveDistillationLoss(nn.Module):
    """
    Комбинированный loss:
    - Контрастная дистилляция признаков (InfoNCE)
    - Логит дистилляция (KL divergence)
    - Classification loss (Cross-Entropy)
    
    Args:
        temperature_kd: температура для KL divergence (логиты)
        temperature_contrast: температура для контрастного loss
        alpha: вес контрастного loss
        beta: вес KD loss
        gamma: вес CE loss
    """
    def __init__(self, temperature_kd=4.0, temperature_contrast=0.07, 
                 alpha=0.5, beta=0.3, gamma=0.2):
        super().__init__()
        self.T_kd = temperature_kd
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        
        self.contrastive_loss = ContrastiveLoss(temperature=temperature_contrast)
        self.kl_loss = nn.KLDivLoss(reduction='batchmean')
        self.ce_loss = nn.CrossEntropyLoss()
        
    def forward(self, student_features, teacher_features, 
                student_logits, teacher_logits, labels):
        """
        Returns: total_loss, contrast_loss, kd_loss, ce_loss
        """
        # 1. Контрастная дистилляция признаков
        contrast_loss = self.contrastive_loss(student_features, teacher_features)
        
        # 2. KL дистилляция логитов
        kd_loss = self.kl_loss(
            F.log_softmax(student_logits / self.T_kd, dim=-1),
            F.softmax(teacher_logits / self.T_kd, dim=-1)
        ) * (self.T_kd ** 2)
        
        # 3. Classification loss
        ce_loss = self.ce_loss(student_logits, labels)
        
        # Комбинированный loss
        total_loss = (self.alpha * contrast_loss + 
                     self.beta * kd_loss + 
                     self.gamma * ce_loss)
        
        return total_loss, contrast_loss, kd_loss, ce_loss

# ==============================
# 4. PROJECTION HEAD (для контрастного обучения)
# ==============================
class ProjectionHead(nn.Module):
    """
    Projection head для преобразования признаков в единое пространство
    для контрастного обучения.
    """
    def __init__(self, input_dim, hidden_dim=512, output_dim=256):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        return self.projection(x)

# ==============================
# 5. НАСТРОЙКИ И ИНИЦИАЛИЗАЦИЯ
# ==============================
# Конфиг
ROOT_DIR = 'kvasir-dataset-v2'
BATCH_SIZE = 16
NUM_WORKERS = 0
KD_LR = 1e-4
KD_EPOCHS = 10

# Гиперпараметры контрастной дистилляции
TEMPERATURE_KD = 4.0
TEMPERATURE_CONTRAST = 0.07
ALPHA_CONTRAST = 0.5  # вес контрастного loss
BETA_KD = 0.3         # вес KL loss
GAMMA_CE = 0.2        # вес CE loss

SAVE_BEST = True



Device: cuda


In [ ]:
from core.vision_encoder import pe
from core.vision_encoder import transforms

print('Загружаем модель-учителя PE-Core-L14-336...')
teacher_model = pe.CLIP.from_config('PE-Core-L14-336', pretrained=True).to(device).float().eval()

print('Загружаем модель-студента PE-Core-S16-384...')
student_model = pe.CLIP.from_config('PE-Core-S16-384', pretrained=True).to(device).float()

teacher_preprocessor = transforms.get_image_transform(teacher_model.image_size)
student_preprocessor = transforms.get_image_transform(student_model.image_size)
teacher_dim = teacher_model.visual.output_dim
student_dim = student_model.visual.output_dim


# Загрузка моделей (используйте ваш код из секции 2)
# ...existing code...
# teacher_model, student_model, teacher_preprocessor, student_preprocessor
# teacher_dim, student_dim

# ==============================
# 6. СОЗДАНИЕ PROJECTION HEADS
# ==============================
# Приводим признаки к единому пространству для контрастного обучения
projection_dim = 256

teacher_projection = ProjectionHead(teacher_dim, hidden_dim=512, output_dim=projection_dim).to(device)
student_projection = ProjectionHead(student_dim, hidden_dim=512, output_dim=projection_dim).to(device)

print(f'✓ Projection heads созданы: {teacher_dim}/{student_dim} -> {projection_dim}')

In [ ]:
# ==============================
# 7. ИНИЦИАЛИЗАЦИЯ LOSS И OPTIMIZER
# ==============================
criterion = ContrastiveDistillationLoss(
    temperature_kd=TEMPERATURE_KD,
    temperature_contrast=TEMPERATURE_CONTRAST,
    alpha=ALPHA_CONTRAST,
    beta=BETA_KD,
    gamma=GAMMA_CE
)

# Оптимизатор для студента + projection heads + classification head
optimizer = torch.optim.AdamW(
    list(student_model.parameters()) + 
    list(student_head.parameters()) + 
    list(student_projection.parameters()) +
    list(teacher_projection.parameters()),  # опционально можно заморозить
    lr=KD_LR, 
    weight_decay=1e-5
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=KD_EPOCHS)

print('✓ Контрастная дистилляция инициализирована')

In [ ]:


# ==============================
# 8. TRAINING LOOP
# ==============================
def train_contrastive_distillation(teacher_model, student_model, 
                                   teacher_head, student_head,
                                   teacher_projection, student_projection,
                                   train_loader, val_loader, 
                                   criterion, optimizer, scheduler,
                                   epochs, device):
    
    history = {
        'train_loss': [], 'train_contrast': [], 'train_kd': [], 'train_ce': [],
        'val_loss': [], 'val_contrast': [], 'val_kd': [], 'val_ce': [], 'val_acc': []
    }
    
    best_val_acc = -1.0
    best_state = None
    
    for epoch in range(1, epochs + 1):
        # TRAIN
        student_model.train()
        student_head.train()
        student_projection.train()
        teacher_model.eval()
        teacher_head.eval()
        teacher_projection.eval()
        
        loss_sum = contrast_sum = kd_sum = ce_sum = 0.0
        n_batches = 0
        
        for x_t, x_s, y in train_loader:
            x_t = x_t.to(device, non_blocking=True)
            x_s = x_s.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            
            # Teacher forward (frozen)
            with torch.no_grad():
                t_features = teacher_model.encode_image(x_t)
                t_projected = teacher_projection(t_features)
                t_logits = teacher_head(t_features)
            
            # Student forward
            s_features = student_model.encode_image(x_s)
            s_projected = student_projection(s_features)
            s_logits = student_head(s_features)
            
            # Loss calculation
            total_loss, contrast_loss, kd_loss, ce_loss = criterion(
                s_projected, t_projected,
                s_logits, t_logits, y
            )
            
            # Backward
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
            
            loss_sum += total_loss.item()
            contrast_sum += contrast_loss.item()
            kd_sum += kd_loss.item()
            ce_sum += ce_loss.item()
            n_batches += 1
        
        avg_train_loss = loss_sum / max(1, n_batches)
        avg_train_contrast = contrast_sum / max(1, n_batches)
        avg_train_kd = kd_sum / max(1, n_batches)
        avg_train_ce = ce_sum / max(1, n_batches)
        
        # VALIDATION
        val_loss, val_contrast, val_kd, val_ce, val_acc = validate_contrastive(
            teacher_model, student_model,
            teacher_head, student_head,
            teacher_projection, student_projection,
            val_loader, criterion, device
        )
        
        scheduler.step()
        
        # Save history
        history['train_loss'].append(avg_train_loss)
        history['train_contrast'].append(avg_train_contrast)
        history['train_kd'].append(avg_train_kd)
        history['train_ce'].append(avg_train_ce)
        history['val_loss'].append(val_loss)
        history['val_contrast'].append(val_contrast)
        history['val_kd'].append(val_kd)
        history['val_ce'].append(val_ce)
        history['val_acc'].append(val_acc)
        
        print(f'Epoch {epoch}/{epochs}')
        print(f'  Train: loss={avg_train_loss:.4f} contrast={avg_train_contrast:.4f} kd={avg_train_kd:.4f} ce={avg_train_ce:.4f}')
        print(f'  Val:   loss={val_loss:.4f} contrast={val_contrast:.4f} kd={val_kd:.4f} ce={val_ce:.4f} acc={val_acc:.2f}%')
        
        # Save best model
        if SAVE_BEST and val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {
                'student_model': student_model.state_dict(),
                'student_head': student_head.state_dict(),
                'student_projection': student_projection.state_dict(),
                'epoch': epoch,
                'val_acc': val_acc
            }
            torch.save(best_state, 'student_contrastive_best.pth')
            print(f'  ✓ Сохранена лучшая модель (ValAcc={val_acc:.2f}%)')
    
    return history, best_val_acc



In [ ]:
# ==============================
# 9. VALIDATION FUNCTION
# ==============================
def validate_contrastive(teacher_model, student_model,
                         teacher_head, student_head,
                         teacher_projection, student_projection,
                         loader, criterion, device):
    
    teacher_model.eval()
    student_model.eval()
    teacher_head.eval()
    student_head.eval()
    teacher_projection.eval()
    student_projection.eval()
    
    total_loss = total_contrast = total_kd = total_ce = 0.0
    correct = total = n_batches = 0
    
    with torch.no_grad():
        for x_t, x_s, y in loader:
            x_t = x_t.to(device, non_blocking=True)
            x_s = x_s.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            
            t_features = teacher_model.encode_image(x_t)
            t_projected = teacher_projection(t_features)
            t_logits = teacher_head(t_features)
            
            s_features = student_model.encode_image(x_s)
            s_projected = student_projection(s_features)
            s_logits = student_head(s_features)
            
            loss, contrast, kd, ce = criterion(
                s_projected, t_projected,
                s_logits, t_logits, y
            )
            
            total_loss += loss.item()
            total_contrast += contrast.item()
            total_kd += kd.item()
            total_ce += ce.item()
            
            _, pred = torch.max(s_logits, 1)
            total += y.size(0)
            correct += (pred == y).sum().item()
            n_batches += 1
    
    avg_loss = total_loss / max(1, n_batches)
    avg_contrast = total_contrast / max(1, n_batches)
    avg_kd = total_kd / max(1, n_batches)
    avg_ce = total_ce / max(1, n_batches)
    acc = 100.0 * correct / total if total > 0 else 0.0
    
    return avg_loss, avg_contrast, avg_kd, avg_ce, acc



In [ ]:
# ==============================
# 10. ЗАПУСК ОБУЧЕНИЯ
# ==============================
print('\n' + '='*80)
print('НАЧАЛО КОНТРАСТНОЙ ДИСТИЛЛЯЦИИ')
print('='*80 + '\n')

history, best_val_acc = train_contrastive_distillation(
    teacher_model, student_model,
    teacher_head, student_head,
    teacher_projection, student_projection,
    train_loader, val_loader,
    criterion, optimizer, scheduler,
    KD_EPOCHS, device
)

print(f'\n✓ Обучение завершено. Лучшая ValAcc: {best_val_acc:.2f}%')

# ==============================
# 11. VISUALIZATION
# ==============================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0,0].plot(history['train_loss'], label='Train')
axes[0,0].plot(history['val_loss'], label='Val')
axes[0,0].set_title('Total Loss')
axes[0,0].legend()
axes[0,0].grid(True)

axes[0,1].plot(history['train_contrast'], label='Train')
axes[0,1].plot(history['val_contrast'], label='Val')
axes[0,1].set_title('Contrastive Loss (InfoNCE)')
axes[0,1].legend()
axes[0,1].grid(True)

axes[0,2].plot(history['train_kd'], label='Train')
axes[0,2].plot(history['val_kd'], label='Val')
axes[0,2].set_title('KD Loss')
axes[0,2].legend()
axes[0,2].grid(True)

axes[1,0].plot(history['train_ce'], label='Train')
axes[1,0].plot(history['val_ce'], label='Val')
axes[1,0].set_title('CE Loss')
axes[1,0].legend()
axes[1,0].grid(True)

axes[1,1].plot(history['val_acc'], color='green')
axes[1,1].set_title('Validation Accuracy')
axes[1,1].grid(True)

axes[1,2].axis('off')

plt.tight_layout()
plt.savefig('contrastive_distillation_history.png', dpi=150)
print('📈 График сохранен: contrastive_distillation_history.png')
plt.show()